# 02 - Exploratory Data Analysis (EDA)
**Project:** Food Delivery Operations Analysis
**Goal:** Find what causes delivery delays and recommend how to reduce delivery time.

This notebook loads the cleaned file `data/clean/train_clean.csv` and answers one business question per section.
Charts are saved to `images/` so you can reuse them in your README and dashboard.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (9, 5)
os.makedirs("../images", exist_ok=True)

df = pd.read_csv("../data/clean/train_clean.csv")
df.columns = [c.strip().lower() for c in df.columns]   # lowercase names so lookups are easy
print(df.shape)
df.head()

In [ ]:
# Find columns by keyword, so the code works even if your cleaned names differ slightly
def find(*keys, avoid=("code",)):
    for k in keys:
        for c in df.columns:
            if k in c and not any(a in c for a in avoid):
                return c
    return None

COL = {
    "time":      find("time_taken", "delivery_time", "time_taken(min)"),
    "traffic":   find("traffic"),
    "weather":   find("weather"),
    "city":      find("city"),
    "vehicle":   find("type_of_vehicle", "vehicle_type"),
    "veh_cond":  find("vehicle_condition"),
    "age":       find("age"),
    "rating":    find("rating"),
    "festival":  find("festival"),
    "multi":     find("multiple"),
    "order_type":find("type_of_order", "order_type"),
    "distance":  "distance_km" if "distance_km" in df.columns else None,
    "hour":      "order_hour" if "order_hour" in df.columns else None,
    "dow":       "day_of_week" if "day_of_week" in df.columns else None,
    "slot":      "time_slot" if "time_slot" in df.columns else None,
    "late":      "is_late" if "is_late" in df.columns else None,
}
for k, v in COL.items():
    print(f"{k:11s} -> {v}")

**Check the printout above.** Every line should show a column name. If one shows `None`, that column is missing from your cleaned file and its section below will be skipped automatically.

In [ ]:
T = COL["time"]
# Make sure the target is numeric
df[T] = pd.to_numeric(df[T], errors="coerce")

if COL["late"] is None:
    df["is_late"] = (df[T] > 32).astype(int)
    COL["late"] = "is_late"
L = COL["late"]

# Headline KPIs
kpis = {
    "Total orders": len(df),
    "Average delivery time (min)": round(df[T].mean(), 1),
    "Median delivery time (min)": round(df[T].median(), 1),
    "% late orders (>32 min)": round(df[L].mean() * 100, 1),
}
if COL["rating"]:   kpis["Average rider rating"] = round(df[COL["rating"]].mean(), 2)
if COL["distance"]: kpis["Average distance (km)"] = round(df[COL["distance"]].mean(), 1)
pd.Series(kpis).to_frame("Value")

## 1. Distribution of delivery time

In [ ]:
fig, ax = plt.subplots()
sns.histplot(df[T], bins=40, kde=True, ax=ax, color="#2a6f97")
ax.axvline(df[T].mean(), color="red", ls="--", label=f"Mean {df[T].mean():.1f} min")
ax.axvline(32, color="orange", ls="--", label="Late threshold (32 min)")
ax.set_title("Distribution of delivery time")
ax.set_xlabel("Delivery time (minutes)")
ax.legend()
plt.tight_layout()
plt.savefig("../images/01_delivery_time_distribution.png", dpi=150)
plt.show()

## 2. Which factors drive delivery time? (traffic, weather, city type, vehicle)

In [ ]:
def bar_avg(colkey, title, fname, order=None, rotate=0):
    c = COL[colkey]
    if c is None:
        print(f"Skipped: no '{colkey}' column"); return
    g = (df.groupby(c)[T].agg(["mean", "count"]).reset_index()
           .sort_values("mean", ascending=False))
    if order: g = g.set_index(c).reindex(order).dropna().reset_index()
    fig, ax = plt.subplots()
    sns.barplot(data=g, x=c, y="mean", ax=ax, color="#2a6f97")
    for i, r in g.iterrows():
        ax.text(list(g[c]).index(r[c]), r["mean"] + 0.3, f"{r['mean']:.1f}\n(n={int(r['count'])})",
                ha="center", fontsize=8)
    ax.set_title(title)
    ax.set_ylabel("Average delivery time (min)")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.savefig(f"../images/{fname}", dpi=150)
    plt.show()
    return g

traffic_order = [x for x in ["Low", "Medium", "High", "Jam"] if x in df[COL["traffic"]].astype(str).str.strip().unique()] if COL["traffic"] else None
bar_avg("traffic", "Average delivery time by road traffic", "02_time_by_traffic.png", order=traffic_order)

In [ ]:
bar_avg("weather", "Average delivery time by weather", "03_time_by_weather.png", rotate=20)

In [ ]:
bar_avg("city", "Average delivery time by city type", "04_time_by_city.png")

In [ ]:
bar_avg("vehicle", "Average delivery time by vehicle type", "05_time_by_vehicle.png")

**Note on sample size:** each bar shows `n=`, the number of orders behind it. Small groups (for example Semi-Urban cities) can look extreme but are less reliable. Mention this in your README.

## 3. When are deliveries slowest? (hour, time slot, day of week)

In [ ]:
if COL["hour"]:
    g = df.groupby(COL["hour"])[T].mean().reset_index()
    fig, ax = plt.subplots()
    sns.lineplot(data=g, x=COL["hour"], y=T, marker="o", ax=ax, color="#2a6f97")
    ax.set_title("Average delivery time by order hour")
    ax.set_xlabel("Hour of day (0-23)")
    ax.set_ylabel("Average delivery time (min)")
    ax.set_xticks(range(0, 24))
    plt.tight_layout()
    plt.savefig("../images/06_time_by_hour.png", dpi=150)
    plt.show()
else:
    print("Skipped: no order_hour column")

In [ ]:
if COL["slot"]:
    bar_avg("slot", "Average delivery time by time slot", "07_time_by_slot.png")
if COL["dow"]:
    bar_avg("dow", "Average delivery time by day of week", "08_time_by_day.png", rotate=20)

## 4. Does distance matter?

In [ ]:
if COL["distance"]:
    d = df[(df[COL["distance"]] > 0) & (df[COL["distance"]] < df[COL["distance"]].quantile(0.99))]
    sample = d.sample(min(5000, len(d)), random_state=42)
    fig, ax = plt.subplots()
    sns.regplot(data=sample, x=COL["distance"], y=T, scatter_kws={"alpha": 0.15, "s": 10},
                line_kws={"color": "red"}, ax=ax)
    ax.set_title("Distance vs delivery time (5,000-order sample)")
    ax.set_xlabel("Distance (km)")
    ax.set_ylabel("Delivery time (min)")
    plt.tight_layout()
    plt.savefig("../images/09_distance_vs_time.png", dpi=150)
    plt.show()
    print("Correlation:", round(d[COL["distance"]].corr(d[T]), 3))
else:
    print("Skipped: no distance_km column")

Restaurant coordinates were missing for about 3,640 orders during cleaning, so distance is only calculated for the rest. Say this clearly if you quote a distance finding.

## 5. Rider factors: age, rating, vehicle condition

In [ ]:
if COL["rating"]:
    df["rating_band"] = pd.cut(df[COL["rating"]], bins=[0, 3.9, 4.4, 4.7, 5.0],
                               labels=["<4.0", "4.0-4.4", "4.5-4.7", "4.8-5.0"])
    g = df.groupby("rating_band", observed=True)[T].agg(["mean", "count"]).reset_index()
    fig, ax = plt.subplots()
    sns.barplot(data=g, x="rating_band", y="mean", ax=ax, color="#2a6f97")
    ax.set_title("Average delivery time by rider rating band")
    ax.set_ylabel("Average delivery time (min)")
    plt.tight_layout()
    plt.savefig("../images/10_time_by_rating.png", dpi=150)
    plt.show()
    display(g)

In [ ]:
if COL["age"]:
    df["age_band"] = pd.cut(df[COL["age"]], bins=[0, 24, 29, 34, 100],
                            labels=["<25", "25-29", "30-34", "35+"])
    g = df.groupby("age_band", observed=True)[T].agg(["mean", "count"]).reset_index()
    fig, ax = plt.subplots()
    sns.barplot(data=g, x="age_band", y="mean", ax=ax, color="#2a6f97")
    ax.set_title("Average delivery time by rider age band")
    ax.set_ylabel("Average delivery time (min)")
    plt.tight_layout()
    plt.savefig("../images/11_time_by_age.png", dpi=150)
    plt.show()
    display(g)

In [ ]:
bar_avg("veh_cond", "Average delivery time by vehicle condition (0 = worst, 3 = best)", "12_time_by_vehicle_condition.png")

## 6. Late-order rate: festival, multiple deliveries, traffic, weather

In [ ]:
def late_rate(colkey, title, fname, rotate=0):
    c = COL[colkey]
    if c is None:
        print(f"Skipped: no '{colkey}' column"); return
    g = (df.groupby(c)[L].agg(["mean", "count"]).reset_index()
           .sort_values("mean", ascending=False))
    g["late_pct"] = g["mean"] * 100
    fig, ax = plt.subplots()
    sns.barplot(data=g, x=c, y="late_pct", ax=ax, color="#c1121f")
    for i, (_, r) in enumerate(g.iterrows()):
        ax.text(i, r["late_pct"] + 0.5, f"{r['late_pct']:.0f}%", ha="center", fontsize=9)
    ax.set_title(title)
    ax.set_ylabel("% of orders late (>32 min)")
    plt.xticks(rotation=rotate)
    plt.tight_layout()
    plt.savefig(f"../images/{fname}", dpi=150)
    plt.show()

late_rate("festival", "Late-order rate: festival vs normal days", "13_late_by_festival.png")

In [ ]:
late_rate("multi", "Late-order rate by number of orders carried at once", "14_late_by_multiple.png")

In [ ]:
late_rate("traffic", "Late-order rate by traffic", "15_late_by_traffic.png")

In [ ]:
late_rate("weather", "Late-order rate by weather", "16_late_by_weather.png", rotate=20)

## 7. Which factors matter most? (correlation)

In [ ]:
num = df.select_dtypes(include="number").drop(columns=[c for c in df.columns if "id" == c or c.endswith("_id")], errors="ignore")
# Drop raw coordinate columns, they are not meaningful on their own
num = num.drop(columns=[c for c in num.columns if "latitude" in c or "longitude" in c], errors="ignore")
corr = num.corr()[T].drop(T).sort_values(key=abs, ascending=False)

fig, ax = plt.subplots(figsize=(8, 6))
sns.barplot(x=corr.values, y=corr.index, ax=ax, color="#2a6f97")
ax.set_title("Correlation of numeric columns with delivery time")
ax.set_xlabel("Correlation")
plt.tight_layout()
plt.savefig("../images/17_correlation_with_time.png", dpi=150)
plt.show()
corr.round(3).to_frame("correlation")

## 8. Key findings (fill in after reading your charts)

Write 5 to 6 findings with real numbers from your charts. Example format:

1. **Traffic is the biggest driver.** Jam traffic averages about X minutes vs about Y for low traffic.
2. **Peak hours:** deliveries are slowest around __ (average __ min).
3. **Multiple deliveries** raise the late rate from __% to __%.
4. **Festival days** show a late rate of __% vs __% on normal days.
5. **Distance** has a correlation of __ with delivery time.
6. **Rider rating / age / vehicle condition** have (strong / weak) effect: ______.

## 9. Recommendations (for your README and dashboard)

- Add extra riders in the slowest hour slots and in high-traffic zones.
- Cap how many orders a rider carries at once during peak time.
- Show longer delivery estimates on festival days and in bad weather.
- Prioritise vehicle maintenance if vehicle condition shows a clear effect.

Next: build the Power BI dashboard from `data/clean/train_clean.csv`.